In [11]:
# Importing modules
import pandas as pd
import numpy as np
#import matplotlib.pyplot as plt
#import seaborn
import os
#import os as sns
import nltk
nltk.download('words') # Downloads a list of English words, which can be used for various tasks like checking if a given string is a valid English word.
nltk.download('maxent_ne_chunker') #Downloads the Maximum Entropy Named Entity Chunker, which is a model used for named entity recognition (NER). This tool identifies named entities like people, organizations, and locations from text.
nltk.download('punkt') #Downloads the Punkt sentence tokenizer, which is used to split text into sentences and further tokenize sentences into words. 
nltk.download('averaged_perceptron_tagger') #Downloads the Averaged Perceptron Tagger, which is used for part-of-speech (POS) tagging
import re #provides support for regular expressions in Python. Regular expressions are used for searching, matching, and manipulating strings based on specific patterns
from nltk import pos_tag #Imports the `pos_tag` function from the Natural Language Toolkit (NLTK), which assigns part-of-speech tags to each word in a text. It classifies words into categories like noun, verb, adjective, etc., which is useful for syntactic analysis.
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize #Imports the `word_tokenize` function from NLTK, which splits a text into individual words (or tokens).
from collections import Counter #Imports the `Counter` class from the `collections` module. `Counter` is a dict subclass that is used to count hashable objects. It is particularly useful for counting frequencies of words or tokens in a text.
import gensim #a powerful library for topic modeling
from gensim.utils import simple_preprocess #a convenient method for preprocessing text. It tokenizes text by performing lowercasing, removing punctuation, and optionally removing short tokens or stemming words.
import gensim.corpora as corpora # These are used to create and manage the mappings between words and their integer ids in text analysis.
from gensim.corpora import Dictionary #Imports the `Dictionary` class from Gensim's `corpora` module, which is used to map words from text documents to unique integer ids.
nltk.download('stopwords')
import spacy #open-source library for advanced natural language processing (NLP) tasks for pre-processing
from wordcloud import WordCloud, STOPWORDS
#import matplotlib.pyplot
sw = STOPWORDS
os.chdir('..') #The `chdir('..')` function changes the current working directory to the parent directory.
# Read data into ReviewsOFD
#verbal_reports = pd.read_csv(working_dataset, encoding = 'utf-8')
df = pd.read_csv('K:\Processed_Data_Output_v1.csv', encoding = 'utf-8')
#df = pd.read_csv('K:\Processed_Data_Output_v4.csv')
#ReviewsOFD = ReviewsOFD.Reviews.values.tolist()
#ReviewsOFD = [t.split(',') for t in ReviewsOFD]
# Print head
df.head()

[nltk_data] Downloading package words to
[nltk_data]     C:\Users\bbb4464\AppData\Roaming\nltk_data...
[nltk_data]   Package words is already up-to-date!
[nltk_data] Downloading package maxent_ne_chunker to
[nltk_data]     C:\Users\bbb4464\AppData\Roaming\nltk_data...
[nltk_data]   Package maxent_ne_chunker is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\bbb4464\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\bbb4464\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\bbb4464\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
C:\Users\bbb4464\AppData\Local\Temp\ipykernel_4952\403516253.py:30: DtypeWarning: Columns (7) have mixed types. Specify dtype option on i

,UID,OBSR ID,Step Number,Keyword,Verbal Report,Original Statement,Reasoning,Grade
0,1001,1,1,Customer Expectations,The individual had expectations about the deli...,An estimated delivery time of max 35 min ended...,The individual had a clear expectation about t...,-100.0
1,1002,1,2,Perceived Quality,The individual experienced many issues with th...,It was great when it worked until it didn't. ....,The individual had a negative experience with ...,-100.0
2,1003,1,3,Perceived Value,The individual did not receive their order and...,One of the worst experiences with a helpdesk I...,The individual did not receive the value they ...,-100.0
3,1004,1,4,Perceived Image,The individual had a negative experience with ...,One of the worst experiences with a helpdesk I...,The individual's negative experience with the ...,-100.0
4,1005,1,5,Customer Satisfaction,The individual was extremely dissatisfied with...,One of the worst experiences with a helpdesk I...,The individual's experience was extremely nega...,-100.0


In [12]:
import pandas as pd

# Ensure necessary NLTK datasets are downloaded
#nltk.download('stopwords')

# File path for your CSV
file_path = r'K:\Customer_Satisfaction.csv'
df = pd.read_csv(file_path)

# Handle missing values by filling them with empty strings
df['Verbal_Report_corrected'] = df['Verbal_Report_corrected'].fillna('')

# Remove punctuation and convert text to lowercase
df['Verbal_Report_processed'] = df['Verbal_Report_corrected'].apply(lambda x: re.sub('[,\.!?]', '', str(x).lower()))

# Print out the first few rows of the processed column for quick verification
print(df['Verbal_Report_processed'].head())

# Assuming 'Verbal_Report_corrected' is another column containing textual data.
data = df['Verbal_Report_processed']

# Build bigrams and trigrams capture word patterns and common expressions
bigram = gensim.models.Phrases(data, min_count=20, threshold=100)
trigram = gensim.models.Phrases(bigram[data], threshold=100)
bigram_mod = gensim.models.phrases.Phraser(bigram)
trigram_mod = gensim.models.phrases.Phraser(trigram)

# Use only the tagger in Spacy for faster processing A spaCy NLP model is loaded (`en_core_web_sm`) with the parser and named entity recognizer disabled. This speeds up processing when POS tagging and lemmatization are primarily required
nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])

# Define stopwords using NLTK
stop_words = set(stopwords.words('english'))
#The text is lemmatized using spaCy, which reduces words to their base form while retaining specific parts of speech tags (`NOUN`, `ADJ`, `VERB`, `ADV`).
def process_words(texts, stop_words=stop_words, allowed_tags=['NOUN', 'ADJ', 'VERB', 'ADV']): #A custom function `process_words()` is defined to process the text
    """Convert a document into a list of lowercase tokens, build bigrams-trigrams, implement lemmatization."""
    # Remove stopwords, short tokens, and letter accents
    texts = [[word for word in simple_preprocess(str(doc), deacc=True, min_len=3) if word not in stop_words] for doc in texts]

    texts_out = []
    
    # Implement lemmatization and filter out unwanted parts of speech tags, Another round of stopword removal and punctuation removal is conducted post-lemmatization
    for sent in texts:
        doc = nlp(" ".join(sent))
        texts_out.append([token.lemma_ for token in doc if token.pos_ in allowed_tags])
    
    # Remove stopwords and short tokens again after lemmatization
    texts_out = [[word for word in simple_preprocess(str(doc), deacc=True, min_len=3) if word not in stop_words] for doc in texts_out]
    
    return texts_out

# Process your text data
data_ready = process_words(data)

0    the verbal report mentions that the app was wo...
1    the verbal report expresses frustration and di...
2    the verbal report mentions that the individual...
3    the verbal report mentions that the service is...
4    the verbal report mentions that the individual...
Name: Verbal_Report_processed, dtype: object


In [17]:
from sklearn.model_selection import train_test_split 
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

# Step 1: Filter only for Step Number 6 (Customer Loyalty)
df_loyalty = df[df['Step Number'] == 4].copy()

# Step 2: Clean grade column and convert to numeric (handle 999 as missing or neutral)
df_loyalty['Grade'] = pd.to_numeric(df_loyalty['Grade'], errors='coerce')
df_loyalty = df_loyalty.dropna(subset=['Grade'])

# Step 3: Create binary labels: 1 = Loyal (Grade > 0), 0 = Not Loyal or Neutral
df_loyalty['Loyalty_Label'] = df_loyalty['Grade'].apply(lambda x: 1 if x > 0 else 0)

# Step 4: Combine text fields for richer context
df_loyalty['Verbal_Report_corrected'] = df_loyalty['Verbal_Report_corrected'].fillna('')
df_loyalty['Original_Statement_ALL'] = df_loyalty['Original_Statement_ALL'].fillna('')
df_loyalty['Reasoning_All'] = df_loyalty['Reasoning_All'].fillna('')

# Merge all relevant text into a single feature
df_loyalty['Combined_Text'] = (
    df_loyalty['Verbal_Report_corrected'] + ' ' +
    df_loyalty['Original_Statement_ALL'] + ' ' +
    df_loyalty['Reasoning_All']
)

# Step 5: Train-test split
texts = df_loyalty['Combined_Text'].astype(str).tolist()
labels = df_loyalty['Loyalty_Label'].tolist()

X_train, X_test, y_train, y_test = train_test_split(texts, labels, test_size=0.2, random_state=42)

# Step 6: Build pipeline and classify
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=1000)),
    ('clf', RandomForestClassifier(n_estimators=100, random_state=42))
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

# Step 7: Print performance
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.94      0.92       617
           1       0.89      0.85      0.87       378

    accuracy                           0.90       995
   macro avg       0.90      0.89      0.90       995
weighted avg       0.90      0.90      0.90       995



In [23]:
# Importing modules
import pandas as pd
import numpy as np
#import matplotlib.pyplot as plt
#import seaborn
import os
#import os as sns
import nltk
nltk.download('words') # Downloads a list of English words, which can be used for various tasks like checking if a given string is a valid English word.
nltk.download('maxent_ne_chunker') #Downloads the Maximum Entropy Named Entity Chunker, which is a model used for named entity recognition (NER). This tool identifies named entities like people, organizations, and locations from text.
nltk.download('punkt') #Downloads the Punkt sentence tokenizer, which is used to split text into sentences and further tokenize sentences into words. 
nltk.download('averaged_perceptron_tagger') #Downloads the Averaged Perceptron Tagger, which is used for part-of-speech (POS) tagging
import re #provides support for regular expressions in Python. Regular expressions are used for searching, matching, and manipulating strings based on specific patterns
from nltk import pos_tag #Imports the `pos_tag` function from the Natural Language Toolkit (NLTK), which assigns part-of-speech tags to each word in a text. It classifies words into categories like noun, verb, adjective, etc., which is useful for syntactic analysis.
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize #Imports the `word_tokenize` function from NLTK, which splits a text into individual words (or tokens).
from collections import Counter #Imports the `Counter` class from the `collections` module. `Counter` is a dict subclass that is used to count hashable objects. It is particularly useful for counting frequencies of words or tokens in a text.
import gensim #a powerful library for topic modeling
from gensim.utils import simple_preprocess #a convenient method for preprocessing text. It tokenizes text by performing lowercasing, removing punctuation, and optionally removing short tokens or stemming words.
import gensim.corpora as corpora # These are used to create and manage the mappings between words and their integer ids in text analysis.
from gensim.corpora import Dictionary #Imports the `Dictionary` class from Gensim's `corpora` module, which is used to map words from text documents to unique integer ids.
nltk.download('stopwords')
import spacy #open-source library for advanced natural language processing (NLP) tasks for pre-processing
from wordcloud import WordCloud, STOPWORDS
#import matplotlib.pyplot
sw = STOPWORDS
os.chdir('..') #The `chdir('..')` function changes the current working directory to the parent directory.
# Read data into ReviewsOFD
#verbal_reports = pd.read_csv(working_dataset, encoding = 'utf-8')
df = pd.read_csv('K:\Processed_Data_Output_v1.csv', encoding = 'utf-8')
#df = pd.read_csv('K:\Processed_Data_Output_v4.csv')
#ReviewsOFD = ReviewsOFD.Reviews.values.tolist()
#ReviewsOFD = [t.split(',') for t in ReviewsOFD]
# Print head
df.head()

[nltk_data] Downloading package words to
[nltk_data]     C:\Users\bbb4464\AppData\Roaming\nltk_data...
[nltk_data]   Package words is already up-to-date!
[nltk_data] Downloading package maxent_ne_chunker to
[nltk_data]     C:\Users\bbb4464\AppData\Roaming\nltk_data...
[nltk_data]   Package maxent_ne_chunker is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\bbb4464\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\bbb4464\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\bbb4464\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
C:\Users\bbb4464\AppData\Local\Temp\ipykernel_4952\403516253.py:30: DtypeWarning: Columns (7) have mixed types. Specify dtype option on i

,ID,OBSR ID,Step Number,Keyword,Verbal Report,Original Statement,Reasoning,Grade
0,1,sec_0,1,Customer Expectations,The individual had expectations about the deli...,An estimated delivery time of max 35 min ended...,The individual had a clear expectation about t...,-100.0
1,1,sec_0,2,Perceived Quality,The individual experienced many issues with th...,It was great when it worked until it didn't. ....,The individual had a negative experience with ...,-100.0
2,1,sec_0,3,Perceived Value,The individual did not receive their order and...,One of the worst experiences with a helpdesk I...,The individual did not receive the value they ...,-100.0
3,1,sec_0,4,Perceived Image,The individual had a negative experience with ...,One of the worst experiences with a helpdesk I...,The individual's negative experience with the ...,-100.0
4,1,sec_0,5,Customer Satisfaction,The individual was extremely dissatisfied with...,One of the worst experiences with a helpdesk I...,The individual's experience was extremely nega...,-100.0


In [38]:
import pandas as pd

# Ensure necessary NLTK datasets are downloaded
#nltk.download('stopwords')

# File path for your CSV
file_path = r'K:\C_Loy.csv'
df = pd.read_csv(file_path)

# Handle missing values by filling them with empty strings
df['Original Statement'] = df['Original Statement'].fillna('')

# Remove punctuation and convert text to lowercase
df['Original_Statement_processed'] = df['Original Statement'].apply(lambda x: re.sub('[,\.!?]', '', str(x).lower()))

# Print out the first few rows of the processed column for quick verification
print(df['Original_Statement_processed'].head())

# Assuming 'Verbal_Report_corrected' is another column containing textual data.
data = df['Original_Statement_processed']

# Build bigrams and trigrams capture word patterns and common expressions
bigram = gensim.models.Phrases(data, min_count=20, threshold=100)
trigram = gensim.models.Phrases(bigram[data], threshold=100)
bigram_mod = gensim.models.phrases.Phraser(bigram)
trigram_mod = gensim.models.phrases.Phraser(trigram)

# Use only the tagger in Spacy for faster processing A spaCy NLP model is loaded (`en_core_web_sm`) with the parser and named entity recognizer disabled. This speeds up processing when POS tagging and lemmatization are primarily required
nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])

# Define stopwords using NLTK
stop_words = set(stopwords.words('english'))
#The text is lemmatized using spaCy, which reduces words to their base form while retaining specific parts of speech tags (`NOUN`, `ADJ`, `VERB`, `ADV`).
def process_words(texts, stop_words=stop_words, allowed_tags=['NOUN', 'ADJ', 'VERB', 'ADV']): #A custom function `process_words()` is defined to process the text
    """Convert a document into a list of lowercase tokens, build bigrams-trigrams, implement lemmatization."""
    # Remove stopwords, short tokens, and letter accents
    texts = [[word for word in simple_preprocess(str(doc), deacc=True, min_len=3) if word not in stop_words] for doc in texts]

    texts_out = []
    
    # Implement lemmatization and filter out unwanted parts of speech tags, Another round of stopword removal and punctuation removal is conducted post-lemmatization
    for sent in texts:
        doc = nlp(" ".join(sent))
        texts_out.append([token.lemma_ for token in doc if token.pos_ in allowed_tags])
    
    # Remove stopwords and short tokens again after lemmatization
    texts_out = [[word for word in simple_preprocess(str(doc), deacc=True, min_len=3) if word not in stop_words] for doc in texts_out]
    
    return texts_out

# Process your text data
data_ready = process_words(data)

0    last time me and my wife have used uber eats a...
1          i just started using this app for a few day
2                             very upset with this app
3    there are a lot of issues with the app some of...
4                    worst food delivery app ever used
Name: Original_Statement_processed, dtype: object


In [39]:
from sklearn.model_selection import train_test_split 
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
import pandas as pd

# Make numeric safely
df['Step Number'] = pd.to_numeric(df['Step Number'], errors='coerce')
df['Grade'] = pd.to_numeric(df['Grade'], errors='coerce')

# If only one step exists, use it automatically
only_steps = df['Step Number'].dropna().unique()
print("Steps in file:", only_steps)

if len(only_steps) == 1:
    LOYALTY_STEP = int(only_steps[0])
    print(f"Only one step found. Using Step {LOYALTY_STEP}.")
else:
    LOYALTY_STEP = 8  # change if needed

df_loyalty = df[df['Step Number'] == LOYALTY_STEP].copy()

# Remove missing grades
df_loyalty = df_loyalty[df_loyalty['Grade'] != 999]
df_loyalty = df_loyalty.dropna(subset=['Grade'])

print("Rows after filtering:", len(df_loyalty))

if df_loyalty.empty:
    raise ValueError("Still empty after filtering — loyalty likely stored in another column.")

# Label
df_loyalty['Loyalty_Label'] = (df_loyalty['Grade'] > 0).astype(int)

# Combine text
for col in ['Verbal Report', 'Original Statement', 'Reasoning']:
    if col not in df_loyalty.columns:
        df_loyalty[col] = ''
    df_loyalty[col] = df_loyalty[col].fillna('')

df_loyalty['Combined_Text'] = (
    df_loyalty['Verbal Report'] + ' ' +
    df_loyalty['Original Statement'] + ' ' +
    df_loyalty['Reasoning']
).str.strip()

texts = df_loyalty['Combined_Text'].tolist()
labels = df_loyalty['Loyalty_Label'].tolist()

X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=1000)),
    ('clf', RandomForestClassifier(n_estimators=100, random_state=42))
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print(classification_report(y_test, y_pred))



Steps in file: [8]
Only one step found. Using Step 8.
Rows after filtering: 21160
              precision    recall  f1-score   support

           0       0.96      0.98      0.97      3226
           1       0.95      0.86      0.90      1006

    accuracy                           0.95      4232
   macro avg       0.95      0.92      0.94      4232
weighted avg       0.95      0.95      0.95      4232

